# Toy Example

This notebook contains the toy example included in the Appendix or our manuscript. We start by constructing a random network (Watts-Strogatz) containing ten nodes. Then, we add four nodes to artificially infuse a smurfing pattern. 

We proceed with visualising the adjacency matrices of two nodes in the network for inclusion in the paper. Afterwards, we calculate the GARG-AML scores. 

In [ ]:
import os
DIR = "../"
os.chdir(DIR)

## Network construction

In [ ]:
import pandas as pd
import igraph as ig
import matplotlib.pyplot as plt

In [ ]:
import random
graph_dir = ig.Graph(directed=True)
n = 20
random.seed(1997)
rg = graph_dir.Watts_Strogatz(1, n, 2, 0.2)

In [ ]:
nodes = []
for i in range(n+4):
    nodes.append(i)
sources = []
targets = []
for i in rg.es:
    sources.append(i.source)
    targets.append(i.target)

targets.append(n)
targets.append(n+1)
targets.append(n+2)
sources.append(n+3)
sources.append(n+3)
sources.append(n+3)

targets.append(n-1)
targets.append(n-1)
targets.append(n-1)
sources.append(n)
sources.append(n+1)
sources.append(n+2)


In [ ]:
graph_dir.add_vertices(nodes)
graph_dir.add_edges(zip(sources, targets))
ig.plot(graph_dir)

## Undirected

In [ ]:
graph = ig.Graph(directed=False)
graph.add_vertices(nodes)
graph.add_edges(zip(sources, targets))

layout_fr = graph.layout("fr")

visual_style = {}
visual_style["vertex_color"] = "Cornflower Blue"
visual_style["vertex_size"] = 23
visual_style["vertex_label"] = graph.vs["name"]
visual_style["vertex_label_color"] = "white"
visual_style["edge_arrow_size"] = 0.5
visual_style["layout"] = layout_fr
visual_style["bbox"] = (550, 400)

#ig.plot(graph,"FigureA1.pdf", **visual_style)


In [ ]:
#ig.plot(graph_dir,"FigureA4.pdf", **visual_style)

In [ ]:
import networkx as nx
from src.methods.utils.neighbourhood_functions import GARG_AML_nodeselection
G_copy = nx.from_edgelist(zip(sources, targets), create_using=nx.Graph)
G_copy_dir = nx.from_edgelist(zip(sources, targets), create_using=nx.DiGraph)
G_copy_undir = G_copy_dir.to_undirected()
G_copy_rev = G_copy_dir.reverse(copy=True)

In [ ]:
node = 23
G_ego_second = nx.ego_graph(G_copy, node, 2)
G_ego_second_und = nx.ego_graph(G_copy_undir, node, 2) #Use both incoming and outgoing edges
G_ego_second_dir = nx.subgraph(G_copy_dir, G_ego_second_und.nodes)
G_ego_second_rev = nx.ego_graph(G_copy_rev, node, 2) #Look at the reverse graph to get the incoming edges

# nodes_ordered are the nodes ordered as node, 2nd order and 1st order neighbours
nodes_1, nodes_2, nodes_ordered = GARG_AML_nodeselection(G_ego_second, node, directed = False)
nodes_0, nodes_1, nodes_2, nodes_ordered_dir = GARG_AML_nodeselection(G_ego_second_dir, node, directed = True, G_ego_second_und = G_ego_second_und, G_ego_second_rev = G_ego_second_rev)

adj_full = nx.adjacency_matrix(G_ego_second, nodelist=nodes_ordered).toarray()
adj_full_dir = nx.adjacency_matrix(G_ego_second_dir, nodelist=nodes_ordered_dir).toarray()

In [ ]:
nodes_ordered_dir

In [ ]:
adj_full_dir

In [ ]:
visual_style = {}
visual_style["vertex_size"] = 23
visual_style["vertex_label"] = graph.vs["name"]
visual_style["vertex_label_color"] = "white"
visual_style["edge_arrow_size"] = 0.5
visual_style["layout"] = layout_fr
visual_style["bbox"] = (550, 400)

# colour the ego node red, the 2nd order neighbours orange and the 1st order neighbours blue
color_dict = {}
for v in graph.vs:
    if v.index == node:
        color_dict[v.index] = "red"
    elif v.index in nodes_2:
        color_dict[v.index] = "orange"
    elif v.index in nodes_1:
        color_dict[v.index] = "Cornflower Blue"
    else:
        color_dict[v.index] = "lightgrey"
visual_style["vertex_color"] = [color_dict[v.index] for v in graph.vs]
ig.plot(graph, "FigureA3.pdf", **visual_style)
#fig.plot(graph, f"toyexample_{node}_undirected.pdf", **visual_style)
ig.plot(graph_dir, "FigureA6.pdf", **visual_style)

In [ ]:
node = n+3
G_ego_second = nx.ego_graph(G_copy, node, 2)
# nodes_ordered are the nodes ordered as node, 2nd order and 1st order neighbours
nodes_1, nodes_2, nodes_ordered = GARG_AML_nodeselection(G_ego_second, node, directed = False)

adj_full = nx.adjacency_matrix(G_ego_second, nodelist=nodes_ordered).toarray()
adj_full

In [ ]:
visual_style = {}
visual_style["vertex_size"] = 23
visual_style["vertex_label"] = graph.vs["name"]
visual_style["vertex_label_color"] = "white"
visual_style["edge_arrow_size"] = 0.5
visual_style["layout"] = layout_fr
visual_style["bbox"] = (550, 400)

# colour the ego node red, the 2nd order neighbours orange and the 1st order neighbours blue
color_dict = {}
for v in graph.vs:
    if v.index == node:
        color_dict[v.index] = "red"
    elif v.index in nodes_2:
        color_dict[v.index] = "orange"
    elif v.index in nodes_1:
        color_dict[v.index] = "Cornflower Blue"
    else:
        color_dict[v.index] = "lightgrey"
visual_style["vertex_color"] = [color_dict[v.index] for v in graph.vs]

ig.plot(graph, "FigureA2.pdf", **visual_style)
#ig.plot(graph, f"toyexample_{node}_undirected.pdf", **visual_style)
ig.plot(graph_dir, "FigureA5.pdf", **visual_style)

# GARG-AML Score Calculation

In [ ]:
from src.methods.GARGAML import GARG_AML_node_undirected_measures, GARG_AML_node_directed_measures

In [ ]:
measure_1, measure_2, measure_3, size_1, size_2, size_3 = GARG_AML_node_undirected_measures(8, G_copy, include_size=True)
print(measure_1, measure_2, measure_3, size_1, size_2, size_3)

total_size = size_1 + size_3
measure = measure_2 - (size_1 * measure_1 + size_3 * measure_3) / total_size
print(f'GARG-AML score: {measure}')

In [ ]:
measure_1, measure_2, measure_3, size_1, size_2, size_3 = GARG_AML_node_undirected_measures(23, G_copy, include_size=True)
print(measure_1, measure_2, measure_3, size_1, size_2, size_3)

total_size = size_1 + size_3
measure = measure_2 - (size_1 * measure_1 + size_3 * measure_3) / total_size
print(f'GARG-AML score: {measure}')

In [ ]:
measure_00, measure_01, measure_02, measure_10, measure_11, measure_12, measure_20, measure_21, measure_22 = GARG_AML_node_directed_measures(23, G_copy_dir, G_copy_undir, G_copy_rev)
print(measure_00, measure_01, measure_02)
print(measure_10, measure_11, measure_12)
print(measure_20, measure_21, measure_22)

In [ ]:
measure_00, measure_01, measure_02, measure_10, measure_11, measure_12, measure_20, measure_21, measure_22 = GARG_AML_node_directed_measures(8, G_copy_dir, G_copy_undir, G_copy_rev)
print(measure_00, measure_01, measure_02)
print(measure_10, measure_11, measure_12)
print(measure_20, measure_21, measure_22)

In [ ]:
import numpy as np
print(np.mean([measure_01, measure_12]))
print(np.mean([measure_00, measure_02, measure_10, measure_11, measure_20, measure_21, measure_22]))

In [ ]:
measure_high = np.mean([measure_01, measure_12])
measure_low = np.mean([measure_10, measure_21, measure_00, measure_02, measure_11, measure_20, measure_22])
measure = measure_high - measure_low; print(measure)